# Phase 06B: Cross-Teacher Evidence Synthesis & Development Conclusion

Phase 06B hoàn tất **multi-teacher confirmation** bằng cách tổng hợp ba thí
nghiệm matched Teacher–Student đã được khóa ở Phase 06A:

- ExpansionNet v2 → S1: Sequence KD, `α = 0,5`;
- BLIP Base → S2: Sequence KD;
- SigLIP ViT-B/16 → S3: Logit KD.

Đơn vị bằng chứng là `cặp × seed`; endpoint chính là
`ΔCIDEr-D = CIDEr-D(KD) − CIDEr-D(baseline cùng Student)`.

Notebook này:

1. xác minh authorization, protocol và manifest bất biến;
2. dựng lại bảng 9 paired runs;
3. tổng hợp theo cặp và toàn bộ ba họ Teacher;
4. áp dụng gate đã đăng ký trước;
5. đóng Phase 6 và mở cổng **Phase 7 mobile deployment benchmark**.

Không huấn luyện, không đọc ảnh/checkpoint và không mở held-out Test. Xác suất
`2⁻⁹` chỉ là mô tả dấu, không được diễn giải như kiểm định độc lập mạnh vì các
run dùng chung dữ liệu và protocol.

## Kaggle inputs

1. Phase 06A Multi-Teacher Confirmation Entry.
2. Phase 04F ExpansionNet v2 → S1 three-seed aggregate.
3. Phase 04M BLIP → S2 three-seed aggregate.
4. Phase 04S SigLIP → S3 three-seed aggregate.

Runtime: CPU, Internet Off; không cần dataset/model/GPU.


## Cell 1 — Authorization, protocol và source discovery


In [1]:
import csv, hashlib, json, math, os, statistics, time
from datetime import datetime, timezone
from pathlib import Path

STARTED = time.perf_counter()
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
NOTEBOOK_ROOT = INPUT_ROOT / "notebooks" if (INPUT_ROOT / "notebooks").is_dir() else INPUT_ROOT
OUT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT",
    "/kaggle/working//phase6_multi_teacher_evidence_synthesis",
))
AUDIT = OUT / "audits"
CONFIG = OUT / "configs"
RESULT = OUT / "results"
REPORT = OUT / "reports"
MANIFEST = OUT / "manifests"
for directory in [OUT, AUDIT, CONFIG, RESULT, REPORT, MANIFEST]:
    directory.mkdir(parents=True, exist_ok=True)

def now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def exactly_one(name):
    matches = sorted(NOTEBOOK_ROOT.rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name} dưới {NOTEBOOK_ROOT}; "
        f"tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()

def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, relative_path, target)
    return target

P6A_PATH = exactly_one("PHASE6A_MULTI_TEACHER_CONFIRMATION_ENTRY_COMPLETE.json")
P6B_AUTH_PATH = exactly_one("PHASE6B_MULTI_TEACHER_EVIDENCE_SYNTHESIS_AUTHORIZED.json")
A_PATH = exactly_one("PHASE4_TEACHER_A_S1_SEQUENCE_KD_THREE_SEED_AGGREGATE_COMPLETE.json")
B_PATH = exactly_one("PHASE4_TEACHER_B_S2_SEQUENCE_KD_THREE_SEED_AGGREGATE_COMPLETE.json")
C_PATH = exactly_one("PHASE4_TEACHER_C_S3_LOGIT_KD_THREE_SEED_AGGREGATE_COMPLETE.json")

P6A, P6B_AUTH, A, B, C = map(
    read_json, [P6A_PATH, P6B_AUTH_PATH, A_PATH, B_PATH, C_PATH]
)
assert P6A["status"] == "complete"
assert P6A["teacher_families"] == 3 and P6A["matched_pairs"] == 3
assert P6A["paired_runs"] == 9 and P6A["protocol_locked"] is True
assert P6A["phase6b_authorized"] is True
assert P6B_AUTH["status"] == "authorized"
assert P6B_AUTH["new_training_authorized"] is False
for marker in [P6A, P6B_AUTH, A, B, C]:
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False

protocol_path = resolve_inside(
    P6A_PATH.parent, P6B_AUTH["protocol"]["relative_path"]
)
assert sha256(protocol_path) == P6B_AUTH["protocol"]["sha256"]
protocol = read_json(protocol_path)
assert protocol["status"] == "locked"
assert protocol["primary_endpoint"].startswith("paired delta CIDEr-D")
assert protocol["seeds"] == [42, 123, 2026]
assert protocol["raw_logit_fusion_across_teachers"].startswith("prohibited")

write_json(AUDIT / "phase6b_cell01_authorization.json", {
    "schema_version": "phase6b_cell01_v1",
    "created_at_utc": now(),
    "status": "passed",
    "inputs": "5 / 5",
    "protocol_locked": True,
    "teacher_families": 3,
    "matched_pairs": 3,
    "paired_runs": 9,
    "training_started": False,
    "dataset_read": False,
    "checkpoint_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 76)
print("PHASE 06B CELL 1 — AUTHORIZATION")
print("=" * 76)
print("Status                    : PASSED")
print("Inputs                    : 5 / 5 FOUND")
print("Protocol                  : LOCKED")
print("Teacher families / pairs  : 3 / 3")
print("Paired runs               : 9")
print("Training / dataset / ckpt : False / False / False")
print("Test Set status           : SEALED")
print("Progress                  : 1 / 5 code cells")


PHASE 06B CELL 1 — AUTHORIZATION
Status                    : PASSED
Inputs                    : 5 / 5 FOUND
Protocol                  : LOCKED
Teacher families / pairs  : 3 / 3
Paired runs               : 9
Training / dataset / ckpt : False / False / False
Test Set status           : SEALED
Progress                  : 1 / 5 code cells


## Cell 2 — Xác minh manifest và aggregate summaries


In [2]:
STARTED = time.perf_counter()

def verify_manifest(marker_path, marker):
    reference = marker["manifest"]["csv"]
    manifest_path = resolve_inside(marker_path.parent, reference["relative_path"])
    assert manifest_path.is_file()
    assert sha256(manifest_path) == reference["sha256"]
    with manifest_path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    expected_rows = marker.get("manifest_rows", reference.get("rows"))
    assert expected_rows is not None and len(rows) == int(expected_rows)
    for row in rows:
        target = resolve_inside(marker_path.parent, row["relative_path"])
        assert target.is_file(), target
        assert target.stat().st_size == int(row["size_bytes"])
        assert sha256(target) == row["sha256"]
    return rows

def verified_summary(marker_path, marker):
    summary_reference = marker["summary"]
    summary_path = resolve_inside(
        marker_path.parent, summary_reference["relative_path"]
    )
    assert summary_path.is_file()
    assert sha256(summary_path) == summary_reference["sha256"]
    summary = read_json(summary_path)
    assert summary["seeds"] == [42, 123, 2026]
    assert summary["selection_split"] == "validation"
    assert summary["test_set_status"] == "sealed"
    assert summary["test_set_used"] is False
    assert len(summary["per_seed"]) == 3
    return summary

p6a_rows = verify_manifest(P6A_PATH, P6A)
a_rows = verify_manifest(A_PATH, A)
b_rows = verify_manifest(B_PATH, B)
c_rows = verify_manifest(C_PATH, C)
A_SUMMARY = verified_summary(A_PATH, A)
B_SUMMARY = verified_summary(B_PATH, B)
C_SUMMARY = verified_summary(C_PATH, C)

for marker in [A, B, C]:
    assert marker["status"] == "complete"
    assert marker["reproduction_status"] == "passed"
    assert marker["seeds"] == [42, 123, 2026]
    assert marker["cider_gate"] == "3 / 3"
    assert marker["protocol_consistency"] == "exact"

verified_rows = len(p6a_rows) + len(a_rows) + len(b_rows) + len(c_rows)
write_json(AUDIT / "phase6b_cell02_integrity.json", {
    "schema_version": "phase6b_cell02_v1",
    "created_at_utc": now(),
    "status": "passed",
    "manifest_rows_verified": verified_rows,
    "aggregate_summaries": "3 / 3 verified",
    "seeds": [42, 123, 2026],
    "protocol_consistency": "exact",
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 76)
print("PHASE 06B CELL 2 — INTEGRITY")
print("=" * 76)
print("Manifest rows verified    :", verified_rows)
print("Aggregate summaries       : 3 / 3 VERIFIED")
print("Seed protocol             : [42, 123, 2026] EXACT")
print("Reproduction status       : 3 / 3 PASSED")
print("Test Set status           : SEALED")
print("Progress                  : 2 / 5 code cells")


PHASE 06B CELL 2 — INTEGRITY
Manifest rows verified    : 46
Aggregate summaries       : 3 / 3 VERIFIED
Seed protocol             : [42, 123, 2026] EXACT
Reproduction status       : 3 / 3 PASSED
Test Set status           : SEALED
Progress                  : 2 / 5 code cells


## Cell 3 — Dựng bảng 9 paired runs và thống kê chéo Teacher


In [3]:
STARTED = time.perf_counter()
specifications = [
    {
        "pair": "A→S1",
        "teacher": "ExpansionNet v2",
        "student": "S1 MobileNetV3-Large",
        "method": "Sequence KD alpha=0.5",
        "summary": A_SUMMARY,
    },
    {
        "pair": "B→S2",
        "teacher": "BLIP Base",
        "student": "S2 MobileViT-S",
        "method": "Sequence KD",
        "summary": B_SUMMARY,
    },
    {
        "pair": "C→S3",
        "teacher": "SigLIP ViT-B/16",
        "student": "S3 TinyViT-5M",
        "method": "Logit KD",
        "summary": C_SUMMARY,
    },
]

per_seed_rows = []
pair_rows = []
for spec in specifications:
    summary = spec["summary"]
    aggregate = summary["aggregate"]
    deltas = []
    for run in summary["per_seed"]:
        row = {
            "pair": spec["pair"],
            "teacher": spec["teacher"],
            "student": spec["student"],
            "method": spec["method"],
            "seed": int(run["seed"]),
            "baseline_cider_d": float(run["baseline_cider_d"]),
            "kd_cider_d": float(run["kd_cider_d"]),
            "delta_cider_d": float(run["delta_cider_d"]),
            "baseline_bleu4": float(run["baseline_bleu_4"]),
            "kd_bleu4": float(run["kd_bleu_4"]),
            "delta_bleu4": float(run["delta_bleu_4"]),
            "baseline_ce": float(run["baseline_validation_ce"]),
            "kd_ce": float(run["kd_validation_ce"]),
            "delta_ce": float(run["delta_validation_ce"]),
        }
        calculated = row["kd_cider_d"] - row["baseline_cider_d"]
        assert math.isclose(calculated, row["delta_cider_d"], abs_tol=1e-12)
        assert row["delta_cider_d"] > 0
        per_seed_rows.append(row)
        deltas.append(row["delta_cider_d"])
    relative_gain = (
        float(aggregate["CIDEr-D"]["paired_delta"]["mean"])
        / float(aggregate["CIDEr-D"]["baseline"]["mean"])
        * 100.0
    )
    pair_rows.append({
        "pair": spec["pair"],
        "teacher": spec["teacher"],
        "student": spec["student"],
        "method": spec["method"],
        "baseline_cider_mean": float(aggregate["CIDEr-D"]["baseline"]["mean"]),
        "kd_cider_mean": float(aggregate["CIDEr-D"]["kd"]["mean"]),
        "delta_cider_mean": float(aggregate["CIDEr-D"]["paired_delta"]["mean"]),
        "delta_cider_sample_std": float(aggregate["CIDEr-D"]["paired_delta"]["sample_std"]),
        "relative_cider_gain_percent": relative_gain,
        "baseline_bleu4_mean": float(aggregate["BLEU-4"]["baseline"]["mean"]),
        "kd_bleu4_mean": float(aggregate["BLEU-4"]["kd"]["mean"]),
        "delta_bleu4_mean": float(aggregate["BLEU-4"]["paired_delta"]["mean"]),
        "baseline_ce_mean": float(aggregate["validation_ce"]["baseline"]["mean"]),
        "kd_ce_mean": float(aggregate["validation_ce"]["kd"]["mean"]),
        "delta_ce_mean": float(aggregate["validation_ce"]["paired_delta"]["mean"]),
        "positive_cider_seeds": sum(value > 0 for value in deltas),
        "seed_count": len(deltas),
    })

assert len(per_seed_rows) == 9
assert {row["seed"] for row in per_seed_rows} == {42, 123, 2026}
assert all(row["positive_cider_seeds"] == 3 for row in pair_rows)
assert all(row["delta_cider_mean"] > 0 for row in pair_rows)
assert all(row["delta_bleu4_mean"] > 0 for row in pair_rows)

all_cider_deltas = [row["delta_cider_d"] for row in per_seed_rows]
all_bleu4_deltas = [row["delta_bleu4"] for row in per_seed_rows]
all_ce_deltas = [row["delta_ce"] for row in per_seed_rows]
macro_cider_gain = statistics.mean(
    row["delta_cider_mean"] for row in pair_rows
)
macro_relative_gain = statistics.mean(
    row["relative_cider_gain_percent"] for row in pair_rows
)
macro_bleu4_gain = statistics.mean(
    row["delta_bleu4_mean"] for row in pair_rows
)
cross_teacher = {
    "schema_version": "phase6b_cross_teacher_statistics_v1",
    "created_at_utc": now(),
    "teacher_families": 3,
    "matched_pairs": 3,
    "paired_runs": 9,
    "positive_cider_runs": sum(value > 0 for value in all_cider_deltas),
    "positive_pair_mean_cider_deltas": sum(
        row["delta_cider_mean"] > 0 for row in pair_rows
    ),
    "positive_pair_mean_bleu4_deltas": sum(
        row["delta_bleu4_mean"] > 0 for row in pair_rows
    ),
    "macro_mean_delta_cider_d": macro_cider_gain,
    "macro_mean_relative_cider_gain_percent": macro_relative_gain,
    "macro_mean_delta_bleu4": macro_bleu4_gain,
    "pooled_mean_delta_cider_d_equal_runs": statistics.mean(all_cider_deltas),
    "pooled_sample_std_delta_cider_d": statistics.stdev(all_cider_deltas),
    "min_pair_mean_delta_cider_d": min(row["delta_cider_mean"] for row in pair_rows),
    "max_pair_mean_delta_cider_d": max(row["delta_cider_mean"] for row in pair_rows),
    "ce_improved_runs": sum(value < 0 for value in all_ce_deltas),
    "one_sided_all_positive_sign_probability_descriptive": 0.5 ** 9,
    "interpretation_warning": (
        "Descriptive only: runs share data/protocol and are not claimed independent; "
        "do not interpret 2^-9 as a strong confirmatory p-value."
    ),
    "selection_split": "validation",
    "test_set_status": "sealed",
    "test_set_used": False,
}

per_seed_csv = RESULT / "phase6b_per_seed_matched_evidence.csv"
with per_seed_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(per_seed_rows[0]))
    writer.writeheader()
    writer.writerows(per_seed_rows)
pair_csv = RESULT / "phase6b_pair_summary.csv"
with pair_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(pair_rows[0]))
    writer.writeheader()
    writer.writerows(pair_rows)
write_json(RESULT / "phase6b_cross_teacher_statistics.json", cross_teacher)
write_json(RESULT / "phase6b_pair_summary.json", {
    "schema_version": "phase6b_pair_summary_v1",
    "created_at_utc": now(),
    "rows": pair_rows,
    "selection_split": "validation",
    "test_set_status": "sealed",
    "test_set_used": False,
})
write_json(AUDIT / "phase6b_cell03_statistics.json", {
    "schema_version": "phase6b_cell03_v1",
    "created_at_utc": now(),
    "status": "passed",
    "positive_cider_runs": "9 / 9",
    "positive_pair_mean_cider_deltas": "3 / 3",
    "positive_pair_mean_bleu4_deltas": "3 / 3",
    "macro_mean_delta_cider_d": macro_cider_gain,
    "macro_mean_relative_cider_gain_percent": macro_relative_gain,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 76)
print("PHASE 06B CELL 3 — CROSS-TEACHER EVIDENCE")
print("=" * 76)
for row in pair_rows:
    print(
        row["pair"],
        "| ΔCIDEr-D", f'{row["delta_cider_mean"]:.6f}',
        "| relative", f'{row["relative_cider_gain_percent"]:.2f}%',
        "| ΔBLEU-4", f'{row["delta_bleu4_mean"]:.6f}',
        "| wins", f'{row["positive_cider_seeds"]}/3',
    )
print("Macro mean ΔCIDEr-D      :", f"{macro_cider_gain:.6f}")
print("Macro relative gain      :", f"{macro_relative_gain:.2f}%")
print("Positive paired runs     : 9 / 9")
print("Progress                 : 3 / 5 code cells")


PHASE 06B CELL 3 — CROSS-TEACHER EVIDENCE
A→S1 | ΔCIDEr-D 0.120607 | relative 12.23% | ΔBLEU-4 0.035208 | wins 3/3
B→S2 | ΔCIDEr-D 0.027295 | relative 2.75% | ΔBLEU-4 0.006934 | wins 3/3
C→S3 | ΔCIDEr-D 0.020414 | relative 1.90% | ΔBLEU-4 0.009513 | wins 3/3
Macro mean ΔCIDEr-D      : 0.056105
Macro relative gain      : 5.63%
Positive paired runs     : 9 / 9
Progress                 : 3 / 5 code cells


## Cell 4 — Áp dụng gate và viết kết luận phát triển


In [4]:
STARTED = time.perf_counter()
gate = protocol["confirmation_gate"]
gate_checks = {
    "teacher_families_3_of_3": len(pair_rows) == 3,
    "paired_runs_9_of_9": len(per_seed_rows) == 9,
    "positive_cider_runs_9_of_9": all(
        row["delta_cider_d"] > 0 for row in per_seed_rows
    ),
    "positive_pair_mean_cider_3_of_3": all(
        row["delta_cider_mean"] > 0 for row in pair_rows
    ),
    "positive_pair_mean_bleu4_3_of_3": all(
        row["delta_bleu4_mean"] > 0 for row in pair_rows
    ),
    "protocol_consistency_exact": all(
        marker["protocol_consistency"] == "exact" for marker in [A, B, C]
    ),
    "test_remains_sealed": all(
        marker["test_set_status"] == "sealed"
        and marker["test_set_used"] is False
        for marker in [P6A, P6B_AUTH, A, B, C]
    ),
}
assert gate["positive_cider_runs"] == "9 / 9"
assert gate["positive_pair_mean_deltas"] == "3 / 3"
assert all(gate_checks.values()), gate_checks

strongest_gain = max(pair_rows, key=lambda row: row["delta_cider_mean"])
most_stable = min(pair_rows, key=lambda row: row["delta_cider_sample_std"])
conclusion = {
    "schema_version": "phase6b_development_conclusion_v1",
    "created_at_utc": now(),
    "status": "passed",
    "decision": "multi_teacher_matched_pair_confirmation_passed",
    "gate_checks": gate_checks,
    "primary_endpoint": "paired delta CIDEr-D on Validation",
    "evidence": {
        "positive_cider_runs": "9 / 9",
        "positive_pair_mean_cider_deltas": "3 / 3",
        "positive_pair_mean_bleu4_deltas": "3 / 3",
        "macro_mean_delta_cider_d": cross_teacher["macro_mean_delta_cider_d"],
        "macro_mean_relative_cider_gain_percent": (
            cross_teacher["macro_mean_relative_cider_gain_percent"]
        ),
        "macro_mean_delta_bleu4": cross_teacher["macro_mean_delta_bleu4"],
        "descriptive_sign_probability": (
            cross_teacher["one_sided_all_positive_sign_probability_descriptive"]
        ),
    },
    "strongest_absolute_cider_gain_pair": strongest_gain["pair"],
    "strongest_absolute_cider_gain": strongest_gain["delta_cider_mean"],
    "most_stable_cider_delta_pair": most_stable["pair"],
    "most_stable_cider_delta_sample_std": most_stable["delta_cider_sample_std"],
    "ce_interpretation": (
        "CE is diagnostic and channel-dependent; generation metrics are the "
        "primary evidence, so CE is not used as a veto gate."
    ),
    "scientific_scope": (
        "The result supports robustness across three heterogeneous matched "
        "Teacher-Student families on Validation. It does not establish that raw "
        "logits can be fused, does not rank unlike Students by raw score, and "
        "does not constitute final held-out Test evidence."
    ),
    "new_training_performed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
CONCLUSION_PATH = RESULT / "phase6b_development_conclusion.json"
write_json(CONCLUSION_PATH, conclusion)

report_text = f"""# Phase 06B — Kết luận Multi-Teacher trên Validation

## Quyết định

**PASSED:** cả 3/3 cặp Teacher–Student và 9/9 paired seed runs đều có
ΔCIDEr-D dương so với baseline cùng Student.

## Bằng chứng chính

| Cặp | Phương pháp | ΔCIDEr-D trung bình | Tăng tương đối | SD ΔCIDEr-D | ΔBLEU-4 |
|---|---|---:|---:|---:|---:|
""" + "\n".join(
    f'| {row["pair"]} | {row["method"]} | {row["delta_cider_mean"]:.6f} | '
    f'{row["relative_cider_gain_percent"]:.2f}% | '
    f'{row["delta_cider_sample_std"]:.6f} | {row["delta_bleu4_mean"]:.6f} |'
    for row in pair_rows
) + f"""

- Macro mean ΔCIDEr-D: **{cross_teacher["macro_mean_delta_cider_d"]:.6f}**.
- Macro mean tăng CIDEr-D tương đối: **{cross_teacher["macro_mean_relative_cider_gain_percent"]:.2f}%**.
- Macro mean ΔBLEU-4: **{cross_teacher["macro_mean_delta_bleu4"]:.6f}**.
- Cặp có mức tăng CIDEr-D tuyệt đối lớn nhất: **{strongest_gain["pair"]}**.
- Cặp có ΔCIDEr-D ổn định nhất qua seed: **{most_stable["pair"]}**.

## Giới hạn diễn giải

`2^-9 = {cross_teacher["one_sided_all_positive_sign_probability_descriptive"]:.9f}`
chỉ là mô tả dấu. Các run dùng chung dữ liệu/protocol nên không tuyên bố là
chín quan sát độc lập. CE là chỉ số chẩn đoán, không thay thế CIDEr-D/BLEU-4.
Không cộng logits giữa Teacher, không xếp hạng raw score giữa Student khác kiến
trúc và chưa dùng held-out Test.

## Bước kế tiếp

Phase 7 benchmark triển khai di động sẽ đánh giá cả ba Student đã KD theo kích
thước mô hình, RAM đỉnh, độ trễ, thông lượng và tính đúng đắn xuất mô hình.
"""
REPORT_PATH = REPORT / "phase6b_development_conclusion.md"
REPORT_PATH.write_text(report_text, encoding="utf-8")

write_json(AUDIT / "phase6b_cell04_conclusion.json", {
    "schema_version": "phase6b_cell04_v1",
    "created_at_utc": now(),
    "status": "passed",
    "decision": conclusion["decision"],
    "all_gates_passed": True,
    "strongest_gain_pair": strongest_gain["pair"],
    "most_stable_pair": most_stable["pair"],
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 76)
print("PHASE 06B CELL 4 — DEVELOPMENT CONCLUSION")
print("=" * 76)
print("Decision                  : MULTI-TEACHER CONFIRMATION PASSED")
print("Strongest CIDEr gain      :", strongest_gain["pair"])
print("Most stable CIDEr delta   :", most_stable["pair"])
print("Raw teacher-logit fusion  : NOT PERMITTED")
print("Held-out Test             : NOT USED")
print("Progress                  : 4 / 5 code cells")


PHASE 06B CELL 4 — DEVELOPMENT CONCLUSION
Decision                  : MULTI-TEACHER CONFIRMATION PASSED
Strongest CIDEr gain      : A→S1
Most stable CIDEr delta   : C→S3
Raw teacher-logit fusion  : NOT PERMITTED
Held-out Test             : NOT USED
Progress                  : 4 / 5 code cells


## Cell 5 — Đóng Phase 6 và authorize Phase 7 mobile benchmark


In [5]:
STARTED = time.perf_counter()
phase7_authorization = {
    "schema_version": "phase7_mobile_deployment_benchmark_entry_authorized_v1",
    "created_at_utc": now(),
    "status": "authorized",
    "required_next": (
        "Phase 07A mobile deployment benchmark entry, artifact registry and "
        "export-readiness preflight"
    ),
    "objective": (
        "Compare deployability of the three KD Students using model size, "
        "peak RAM, latency, throughput and export correctness."
    ),
    "candidate_policy": "retain all three KD Students for resource-quality Pareto evaluation",
    "candidates": [
        {
            "pair": "A→S1",
            "teacher": "ExpansionNet v2",
            "student": "S1 MobileNetV3-Large",
            "method": "Sequence KD alpha=0.5",
            "source_marker": A_PATH.name,
        },
        {
            "pair": "B→S2",
            "teacher": "BLIP Base",
            "student": "S2 MobileViT-S",
            "method": "Sequence KD",
            "source_marker": B_PATH.name,
        },
        {
            "pair": "C→S3",
            "teacher": "SigLIP ViT-B/16",
            "student": "S3 TinyViT-5M",
            "method": "Logit KD",
            "source_marker": C_PATH.name,
        },
    ],
    "phase7a_scope": [
        "resolve selected KD checkpoint artifacts without loading Test",
        "verify export contracts and shared tokenizer/decoder assets",
        "record parameter count and serialized checkpoint size",
        "define CPU/mobile latency and peak-RAM benchmark protocol",
    ],
    "new_training_authorized": False,
    "held_out_test_opening_authorized": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
PHASE7_AUTH_PATH = OUT / "PHASE7_MOBILE_DEPLOYMENT_BENCHMARK_ENTRY_AUTHORIZED.json"
write_json(PHASE7_AUTH_PATH, phase7_authorization)

tracked_files = (
    sorted(AUDIT.glob("*.json"))
    + sorted(CONFIG.glob("*.json"))
    + sorted(RESULT.glob("*"))
    + sorted(REPORT.glob("*"))
)
manifest_rows = []
for path in tracked_files:
    manifest_rows.append({
        "relative_path": path.relative_to(OUT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    })
manifest_path = MANIFEST / "phase6b_manifest.csv"
with manifest_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["relative_path", "size_bytes", "sha256"]
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
sha_path = MANIFEST / "SHA256SUMS"
sha_path.write_text(
    "".join(
        f'{row["sha256"]}  {row["relative_path"]}\n'
        for row in manifest_rows
    ),
    encoding="utf-8",
)

marker = {
    "schema_version": "phase6_multi_teacher_confirmation_complete_v1",
    "created_at_utc": now(),
    "status": "complete",
    "decision": "multi_teacher_matched_pair_confirmation_passed",
    "teacher_families": 3,
    "matched_pairs": 3,
    "paired_runs": 9,
    "positive_cider_runs": "9 / 9",
    "positive_pair_mean_cider_deltas": "3 / 3",
    "positive_pair_mean_bleu4_deltas": "3 / 3",
    "macro_mean_delta_cider_d": cross_teacher["macro_mean_delta_cider_d"],
    "macro_mean_relative_cider_gain_percent": (
        cross_teacher["macro_mean_relative_cider_gain_percent"]
    ),
    "development_conclusion": {
        "relative_path": CONCLUSION_PATH.relative_to(OUT).as_posix(),
        "sha256": sha256(CONCLUSION_PATH),
    },
    "phase7_entry_authorized": True,
    "phase7_authorization_sha256": sha256(PHASE7_AUTH_PATH),
    "manifest": {
        "csv": {
            "relative_path": manifest_path.relative_to(OUT).as_posix(),
            "sha256": sha256(manifest_path),
            "rows": len(manifest_rows),
        },
        "sha256sums": {
            "relative_path": sha_path.relative_to(OUT).as_posix(),
            "sha256": sha256(sha_path),
        },
    },
    "training_started": False,
    "held_out_test_opening_authorized": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
MARKER_PATH = OUT / "PHASE6_MULTI_TEACHER_CONFIRMATION_COMPLETE.json"
write_json(MARKER_PATH, marker)

print("=" * 76)
print("PHASE 06B FINAL STATUS")
print("=" * 76)
print("Status                    : COMPLETE")
print("Confirmation              : PASSED")
print("Teacher families / runs   : 3 / 9")
print("Positive CIDEr-D runs     : 9 / 9")
print("Positive pair means       : 3 / 3")
print("New training              : NOT AUTHORIZED")
print("Phase 7 mobile benchmark  : AUTHORIZED")
print("Held-out Test opening     : NOT AUTHORIZED")
print("Test Set status           : SEALED")
print("Progress                  : 5 / 5 code cells")


PHASE 06B FINAL STATUS
Status                    : COMPLETE
Confirmation              : PASSED
Teacher families / runs   : 3 / 9
Positive CIDEr-D runs     : 9 / 9
Positive pair means       : 3 / 3
New training              : NOT AUTHORIZED
Phase 7 mobile benchmark  : AUTHORIZED
Held-out Test opening     : NOT AUTHORIZED
Test Set status           : SEALED
Progress                  : 5 / 5 code cells
